# 13 â€” Multi-Wave GuessFactor Distributions

**Goal**: Test whether opponents dodge differently when under multi-wave
pressure. If the GF distribution at wave-break shifts or narrows when
multiple waves are in flight, this is a targetable feature for the MLP.

**Key question**: Is $P(\text{GF} \mid \text{situation, wave\_pressure})$
different from $P(\text{GF} \mid \text{situation})$?

**Features to test**:
- `n_our_waves_in_flight` â€” how many of our waves are active when this wave fires
- `nearest_wave_gap` â€” ticks until our nearest other wave arrives at opponent
- `total_wave_damage` â€” combined damage of all in-flight waves

If the answer is yes, these become inputs to the MLP GF targeting model â€”
a competitive edge no online learner can match (not enough data per battle).

In [ ]:
import sys; sys.path.insert(0, '.')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ks_2samp, entropy
from _loader import build_robot_index, load_stratified

sns.set_style('whitegrid')
%matplotlib inline

## 1. Load Data

In [ ]:
selection = build_robot_index(max_robots=50, battles_per_robot=3, seed=42)
waves = load_stratified('waves.csv', selection)
# Load ticks with only the columns we need to save memory
ticks_raw = load_stratified('ticks.csv', selection, row_frac=1.0)
# Keep only columns needed for GF + wave state at each tick
keep_cols = ['battle_id', 'round', 'tick', 'robot_name',
             'gf_current_at_power_1', 'gf_current_at_power_1_5', 'gf_current_at_power_2',
             'our_wave_distance', 'our_wave_remaining',
             'opponent_wave_distance', 'opponent_wave_remaining', 'opponent_wave_eta',
             'distance', 'opponent_lateral_velocity', 'opponent_fired',
             'ticks_since_we_fired', 'ticks_since_opponent_fired',
             'scan_available']
keep_cols = [c for c in keep_cols if c in ticks_raw.columns]
ticks = ticks_raw[keep_cols].copy()
del ticks_raw
print(f"Waves: {len(waves):,} rows")
print(f"Ticks: {len(ticks):,} rows, {len(keep_cols)} cols")

## 2. Compute Multi-Wave Pressure at Each Wave Fire

For each wave in `waves.csv` (a detected opponent fire from the observer's
perspective), compute how many OTHER waves from the same observer were
in flight at that moment. These are *our* waves â€” bullets we fired that
haven't reached the opponent yet.

Note: `waves.csv` records OPPONENT fires. We need OUR fires to compute
our wave pressure on the opponent. We can reconstruct this from the
same waves.csv viewed from the opponent's perspective, or from ticks.csv
wave-tracking columns.

Since the data is dual-perspective (each battle has both sides), we can
use the opponent's waves.csv as our fire events.

In [ ]:
# For each perspective (battle_id, round, robot_name), build a list of
# opponent fire events (= waves in waves.csv). Then for each such event,
# count how many PREVIOUS opponent waves are still in flight.
#
# A wave fired at tick t with speed s at distance d arrives at t + d/s.
# It's "in flight" during [t, t + d/s].

w = waves.sort_values(['battle_id', 'round', 'robot_name', 'tick']).copy()
w['arrival_tick'] = w['tick'] + w['wave_fire_distance'] / w['wave_bullet_speed']
w['damage'] = np.where(w['wave_bullet_power'] <= 1.0,
                       4.0 * w['wave_bullet_power'],
                       6.0 * w['wave_bullet_power'] - 2.0)

# For each wave, count concurrent in-flight waves from same perspective
# and find nearest wave gap (ticks until nearest other wave arrives)
records = []
for key, grp in w.groupby(['battle_id', 'round', 'robot_name']):
    fire_ticks = grp['tick'].values
    arrival_ticks = grp['arrival_tick'].values
    damages = grp['damage'].values
    speeds = grp['wave_bullet_speed'].values
    distances = grp['wave_fire_distance'].values
    powers = grp['wave_bullet_power'].values
    
    for i in range(len(fire_ticks)):
        t = fire_ticks[i]
        # Count other waves in flight at tick t
        in_flight = (fire_ticks[:i] < t) & (arrival_ticks[:i] > t)
        n_in_flight = int(in_flight.sum())
        total_damage = float(damages[in_flight].sum()) + damages[i]
        
        # Nearest other wave arrival gap
        if n_in_flight > 0:
            other_arrivals = arrival_ticks[:i][in_flight]
            my_arrival = arrival_ticks[i]
            gaps = np.abs(other_arrivals - my_arrival)
            nearest_gap = float(gaps.min())
        else:
            nearest_gap = np.nan
        
        records.append({
            'battle_id': key[0],
            'round': key[1],
            'robot_name': key[2],
            'tick': t,
            'arrival_tick': arrival_ticks[i],
            'bullet_power': powers[i],
            'bullet_speed': speeds[i],
            'fire_distance': distances[i],
            'n_waves_in_flight': n_in_flight,
            'total_wave_damage': total_damage,
            'nearest_wave_gap': nearest_gap,
        })

wf = pd.DataFrame(records)
print(f"Wave events with pressure features: {len(wf):,}")
print(f"\nWave pressure distribution:")
print(wf['n_waves_in_flight'].value_counts().sort_index().head(10))

## 3. Join with Opponent GF at Wave Break

For each wave, find the opponent's GF position at the tick when the wave
arrives. This tells us where the opponent actually WAS when our bullet
passed through â€” i.e., did they successfully dodge?

In [ ]:
# Round arrival_tick to integer for joining with ticks
wf['break_tick'] = wf['arrival_tick'].round().astype(int)

# Choose the GF column that matches the bullet power best
# We'll use gf_current_at_power_2 as our primary (most common power range)
# and also grab power_1 and power_1_5 for power-matched analysis

# Join with ticks to get opponent GF at wave break time
# The ticks.csv is from the OBSERVER's perspective (same robot_name),
# and gf_current_at_power_* gives where the opponent sits in GF space.

merged = wf.merge(
    ticks[['battle_id', 'round', 'tick', 'robot_name',
           'gf_current_at_power_1', 'gf_current_at_power_1_5', 'gf_current_at_power_2',
           'distance']].rename(columns={'tick': 'break_tick'}),
    on=['battle_id', 'round', 'robot_name', 'break_tick'],
    how='inner'
)

# Pick the best GF column based on bullet power
def pick_gf(row):
    p = row['bullet_power']
    if p <= 1.25:
        return row['gf_current_at_power_1']
    elif p <= 1.75:
        return row['gf_current_at_power_1_5']
    else:
        return row['gf_current_at_power_2']

merged['gf_at_break'] = merged.apply(pick_gf, axis=1)

# Drop rows with NaN GF (no scan at break tick)
valid = merged.dropna(subset=['gf_at_break']).copy()
print(f"Waves with valid GF at break: {len(valid):,} / {len(wf):,}")
print(f"\nPressure distribution in valid set:")
print(valid['n_waves_in_flight'].value_counts().sort_index().head(10))

## 4. GF Distributions Under Different Wave Pressure

Compare the GF distribution at wave break for:
- No other waves in flight (n=0)
- 1 other wave (n=1)
- 2+ other waves (nâ‰¥2)

Use KS test to check if distributions differ significantly.

In [ ]:
# Categorize by pressure
valid['pressure'] = pd.cut(valid['n_waves_in_flight'],
                           bins=[-1, 0, 1, 100],
                           labels=['none (0)', 'moderate (1)', 'high (2+)'])

# Overall GF stats by pressure
print("=== GF at wave break by wave pressure ===")
for label in ['none (0)', 'moderate (1)', 'high (2+)']:
    subset = valid[valid['pressure'] == label]['gf_at_break']
    print(f"\n  {label}: n={len(subset):,}")
    print(f"    mean={subset.mean():.4f}  std={subset.std():.4f}")
    print(f"    median={subset.median():.4f}  |mean|={subset.abs().mean():.4f}")

# KS tests
gf_none = valid[valid['pressure'] == 'none (0)']['gf_at_break'].dropna()
gf_mod  = valid[valid['pressure'] == 'moderate (1)']['gf_at_break'].dropna()
gf_high = valid[valid['pressure'] == 'high (2+)']['gf_at_break'].dropna()

print("\n=== KS Tests ===")
if len(gf_none) > 0 and len(gf_mod) > 0:
    ks, p = ks_2samp(gf_none, gf_mod)
    print(f"  none vs moderate: KS={ks:.4f}, p={p:.2e}")
if len(gf_none) > 0 and len(gf_high) > 0:
    ks, p = ks_2samp(gf_none, gf_high)
    print(f"  none vs high:     KS={ks:.4f}, p={p:.2e}")
if len(gf_mod) > 0 and len(gf_high) > 0:
    ks, p = ks_2samp(gf_mod, gf_high)
    print(f"  moderate vs high: KS={ks:.4f}, p={p:.2e}")

In [ ]:
# Plot GF distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Overlaid histograms
bins = np.linspace(-1, 1, 61)
for label, color in [('none (0)', 'blue'), ('moderate (1)', 'orange'), ('high (2+)', 'red')]:
    subset = valid[valid['pressure'] == label]['gf_at_break']
    if len(subset) > 0:
        axes[0].hist(subset, bins=bins, alpha=0.4, density=True, label=label, color=color)
axes[0].set_xlabel('GF at wave break')
axes[0].set_ylabel('Density')
axes[0].set_title('GF Distribution by Wave Pressure')
axes[0].legend()

# 2. KDE overlay
for label, color in [('none (0)', 'blue'), ('moderate (1)', 'orange'), ('high (2+)', 'red')]:
    subset = valid[valid['pressure'] == label]['gf_at_break'].dropna()
    if len(subset) > 100:
        subset.plot.kde(ax=axes[1], label=label, color=color)
axes[1].set_xlabel('GF at wave break')
axes[1].set_title('KDE by Wave Pressure')
axes[1].set_xlim(-1.2, 1.2)
axes[1].legend()

# 3. Std of GF by pressure â€” does the distribution narrow?
gf_std_by_pressure = valid.groupby('pressure')['gf_at_break'].std()
gf_std_by_pressure.plot.bar(ax=axes[2], color=['blue', 'orange', 'red'])
axes[2].set_ylabel('Std of GF at break')
axes[2].set_title('GF Spread by Wave Pressure')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 5. Does Inter-Wave Gap Matter?

For waves with at least one other wave in flight, does the *time gap*
between wave arrivals change the GF distribution? A 3-tick gap means
the opponent must dodge both nearly simultaneously; a 20-tick gap means
they're effectively independent.

In [ ]:
# Filter to waves with pressure (at least 1 other wave in flight)
pressured = valid[valid['n_waves_in_flight'] > 0].copy()
pressured['gap_category'] = pd.cut(
    pressured['nearest_wave_gap'],
    bins=[-1, 5, 15, 100],
    labels=['tight (â‰¤5t)', 'moderate (6-15t)', 'loose (>15t)']
)

print("=== GF at break by inter-wave gap ===")
for label in ['tight (â‰¤5t)', 'moderate (6-15t)', 'loose (>15t)']:
    subset = pressured[pressured['gap_category'] == label]['gf_at_break']
    if len(subset) > 0:
        print(f"  {label}: n={len(subset):,}  mean={subset.mean():.4f}  "
              f"std={subset.std():.4f}  |mean|={subset.abs().mean():.4f}")

# KS: tight vs loose
gf_tight = pressured[pressured['gap_category'] == 'tight (â‰¤5t)']['gf_at_break'].dropna()
gf_loose = pressured[pressured['gap_category'] == 'loose (>15t)']['gf_at_break'].dropna()
if len(gf_tight) > 0 and len(gf_loose) > 0:
    ks, p = ks_2samp(gf_tight, gf_loose)
    print(f"\n  KS tight vs loose: KS={ks:.4f}, p={p:.2e}")

In [ ]:
# Scatter: nearest_wave_gap vs |GF at break| â€” does tight gap reduce GF spread?
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if len(pressured) > 0:
    # Bin by gap, compute GF std per bin
    pressured['gap_bin'] = pd.cut(pressured['nearest_wave_gap'],
                                   bins=np.arange(0, 35, 2))
    gap_stats = pressured.groupby('gap_bin')['gf_at_break'].agg(['std', 'mean', 'count'])
    gap_stats = gap_stats[gap_stats['count'] >= 50]
    
    axes[0].plot(range(len(gap_stats)), gap_stats['std'].values, 'o-')
    axes[0].set_xticks(range(len(gap_stats)))
    axes[0].set_xticklabels([str(x) for x in gap_stats.index], rotation=45, ha='right')
    axes[0].set_xlabel('Inter-wave arrival gap (ticks)')
    axes[0].set_ylabel('GF std at break')
    axes[0].set_title('GF Spread vs Wave Arrival Gap')
    
    axes[1].plot(range(len(gap_stats)), gap_stats['mean'].abs().values, 's-', color='red')
    axes[1].set_xticks(range(len(gap_stats)))
    axes[1].set_xticklabels([str(x) for x in gap_stats.index], rotation=45, ha='right')
    axes[1].set_xlabel('Inter-wave arrival gap (ticks)')
    axes[1].set_ylabel('|Mean GF| at break')
    axes[1].set_title('GF Bias vs Wave Arrival Gap')

plt.tight_layout()
plt.show()

## 6. Per-Bot Sensitivity

Which bots change their dodge behavior the most under multi-wave pressure?
These are bots whose surfer is likely NOT damage-weighted â€” the wave
stacking vulnerable population.

In [ ]:
# Per-bot KS test: no-pressure vs pressure GF distributions
bot_ks = []
for bot, grp in valid.groupby('robot_name'):
    gf_no = grp[grp['n_waves_in_flight'] == 0]['gf_at_break'].dropna()
    gf_yes = grp[grp['n_waves_in_flight'] > 0]['gf_at_break'].dropna()
    if len(gf_no) >= 30 and len(gf_yes) >= 30:
        ks, p = ks_2samp(gf_no, gf_yes)
        bot_ks.append({
            'robot_name': bot,
            'ks_stat': ks,
            'p_value': p,
            'gf_std_no_pressure': gf_no.std(),
            'gf_std_pressure': gf_yes.std(),
            'std_delta': gf_yes.std() - gf_no.std(),
            'n_no': len(gf_no),
            'n_yes': len(gf_yes),
        })

bks = pd.DataFrame(bot_ks).sort_values('ks_stat', ascending=False)
print(f"Bots with enough data: {len(bks)}")
print(f"\nTop 15 bots whose GF distribution changes most under wave pressure:")
print(bks.head(15)[['robot_name', 'ks_stat', 'p_value', 'gf_std_no_pressure',
                     'gf_std_pressure', 'std_delta', 'n_no', 'n_yes']].to_string(index=False))
print(f"\nBots where pressure NARROWS GF spread (std_delta < 0): {(bks['std_delta'] < 0).sum()} / {len(bks)}")
print(f"Bots where pressure WIDENS GF spread (std_delta > 0):  {(bks['std_delta'] > 0).sum()} / {len(bks)}")

## 7. Feature Importance Test

Quick test: does adding multi-wave features improve GF prediction?
Train a small random forest to predict `gf_at_break` bin with and
without the wave-pressure features.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_val_score

# Build feature matrix
feat_base = ['fire_distance', 'bullet_speed', 'bullet_power']
feat_wave = ['n_waves_in_flight', 'nearest_wave_gap', 'total_wave_damage']

# Bin GF into 11 bins for classification
valid_ml = valid.dropna(subset=feat_base + ['gf_at_break']).copy()
valid_ml['nearest_wave_gap'] = valid_ml['nearest_wave_gap'].fillna(999)
valid_ml['gf_bin'] = pd.cut(valid_ml['gf_at_break'], bins=np.linspace(-1, 1, 12),
                             labels=False).astype(int)

# Subsample for speed
if len(valid_ml) > 50000:
    valid_ml = valid_ml.sample(50000, random_state=42)

groups = valid_ml['battle_id'].astype('category').cat.codes
cv = GroupKFold(n_splits=5)

# Model A: base features only
X_base = valid_ml[feat_base].values.astype(np.float32)
y = valid_ml['gf_bin'].values

rf_base = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
scores_base = cross_val_score(rf_base, X_base, y, cv=cv.split(X_base, y, groups),
                               scoring='accuracy')

# Model B: base + wave pressure features
X_wave = valid_ml[feat_base + feat_wave].values.astype(np.float32)
rf_wave = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
scores_wave = cross_val_score(rf_wave, X_wave, y, cv=cv.split(X_wave, y, groups),
                               scoring='accuracy')

print(f"GF bin prediction accuracy (11 bins, random={1/11:.3f}):")
print(f"  Base features only:         {scores_base.mean():.4f} Â± {scores_base.std():.4f}")
print(f"  Base + wave pressure:       {scores_wave.mean():.4f} Â± {scores_wave.std():.4f}")
print(f"  Delta:                      {scores_wave.mean() - scores_base.mean():+.4f}")

# Feature importance from full model
rf_wave.fit(X_wave, y)
feat_names = feat_base + feat_wave
importances = pd.Series(rf_wave.feature_importances_, index=feat_names).sort_values(ascending=False)
print(f"\nFeature importances (base + wave):")
for name, imp in importances.items():
    print(f"  {name:30s} {imp:.4f}")

## 8. Summary

**Findings** (to be filled after execution):

1. Does wave pressure shift the GF distribution? (KS test)
2. Does tighter inter-wave gap narrow the distribution? (std vs gap)
3. Which bots are most sensitive to wave pressure? (per-bot KS)
4. Do wave-pressure features improve GF prediction? (RF accuracy delta)

**If positive**: Add `n_waves_in_flight`, `nearest_wave_gap`,
`total_wave_damage` to the MLP targeting model inputs. This is a
competitive edge â€” no online learner has enough data to condition
on wave pressure within a single battle.